# PillSeek — fine-tune pill matcher + build full index
Run cells top to bottom. Before starting: **Runtime → Change runtime type → T4 GPU**.
Total time: roughly 1.5–2.5 hours. Keep this browser tab open (laptop can sleep? No — tab must stay open, but nothing runs on your laptop).

In [ ]:
# 1) Install (about 1 minute)
!pip -q install open_clip_torch
import torch
print('GPU available:', torch.cuda.is_available())

In [ ]:
# 2) Upload pills.json when the button appears
from google.colab import files
up = files.upload()
assert 'pills.json' in up, 'Please upload the pills.json file'

In [ ]:
# 3) Download pill images (20-50 min). Safe to re-run if interrupted - it skips finished files.
import json, os, re, requests
from concurrent.futures import ThreadPoolExecutor

rows = json.load(open('pills.json'))
if isinstance(rows, dict): rows = rows.get('value', rows)
os.makedirs('imgs', exist_ok=True)
jobs = []
for r in rows:
    label = re.sub(r'-\\d+$', '', r['slug'])[:150]
    for url in r.get('images', [])[:4]:
        fname = url.rsplit('/', 1)[-1]
        if 'placeholder' in fname: continue
        jobs.append((url, label, fname))
print(f'{len(rows)} pills, {len(jobs)} images to fetch')

def fetch(job):
    url, label, fname = job
    d = os.path.join('imgs', re.sub(r'[^a-z0-9-]', '_', label))
    os.makedirs(d, exist_ok=True)
    p = os.path.join(d, fname.replace('/', '_'))
    if os.path.exists(p) and os.path.getsize(p) > 0: return 0
    try:
        b = requests.get(url, timeout=30).content
        open(p, 'wb').write(b)
        return 1
    except Exception:
        return -1

with ThreadPoolExecutor(16) as ex:
    res = list(ex.map(fetch, jobs))
print(f'downloaded {res.count(1)}, cached {res.count(0)}, failed {res.count(-1)}')

In [ ]:
# 4) Fine-tune CLIP on your pills (30-60 min)
import glob, random
import open_clip
from PIL import Image
import torch, torch.nn.functional as F
from torchvision import transforms as T

device = 'cuda'
model, _, preprocess = open_clip.create_model_and_transforms('ViT-B-32', pretrained='openai')
model = model.to(device)

by_label = {}
for d in os.listdir('imgs'):
    fs = glob.glob(f'imgs/{d}/*')
    if len(fs) >= 2: by_label[d] = fs
labels = list(by_label)
print(f'{len(labels)} pills with 2+ images for training')

aug = T.Compose([
    T.RandomResizedCrop(224, scale=(0.6, 1.0)),
    T.RandomRotation(180, fill=255),
    T.ColorJitter(0.3, 0.3, 0.15),
    T.ToTensor(),
    T.Normalize((0.48145466, 0.4578275, 0.40821073), (0.26862954, 0.26130258, 0.27577711)),
])

def load(p):
    try: return aug(Image.open(p).convert('RGB'))
    except Exception: return None

opt = torch.optim.AdamW(model.visual.parameters(), lr=1e-5, weight_decay=0.01)
scaler = torch.cuda.amp.GradScaler()
EPOCHS, PAIRS = 3, 32
model.train()
for ep in range(EPOCHS):
    random.shuffle(labels)
    tot, steps = 0.0, 0
    for start in range(0, len(labels) - PAIRS, PAIRS):
        batch = []
        for lb in labels[start:start + PAIRS]:
            a, b = random.sample(by_label[lb], 2)
            ia, ib = load(a), load(b)
            if ia is None or ib is None: continue
            batch += [ia, ib]
        if len(batch) < 8: continue
        x = torch.stack(batch).to(device)
        with torch.cuda.amp.autocast():
            z = F.normalize(model.encode_image(x), dim=-1)
            sim = z @ z.T / 0.07
            sim.fill_diagonal_(-1e4)
            n = z.shape[0]
            target = torch.arange(n, device=device) ^ 1  # partner index (0<->1, 2<->3, ...)
            loss = F.cross_entropy(sim, target)
        opt.zero_grad()
        scaler.scale(loss).backward()
        scaler.step(opt)
        scaler.update()
        tot += loss.item(); steps += 1
        if steps % 50 == 0: print(f'epoch {ep+1} step {steps}: loss {tot/steps:.3f}')
    print(f'=== epoch {ep+1} done, avg loss {tot/max(steps,1):.3f} ===')
torch.save(model.state_dict(), 'pill_clip_finetuned.pt')
print('saved pill_clip_finetuned.pt')

In [ ]:
# 5) Build the FULL index with the fine-tuned model (20-40 min)
import numpy as np
model.eval()
vecs, meta = [], []
all_files = glob.glob('imgs/*/*')
print(f'indexing {len(all_files)} images')
B = 64
with torch.no_grad():
    for i in range(0, len(all_files), B):
        chunk, kept = [], []
        for p in all_files[i:i+B]:
            try:
                chunk.append(preprocess(Image.open(p).convert('RGB')))
                kept.append(p)
            except Exception: pass
        if not chunk: continue
        x = torch.stack(chunk).to(device)
        with torch.cuda.amp.autocast():
            z = F.normalize(model.encode_image(x), dim=-1)
        vecs.append(z.float().cpu().numpy())
        meta += [{'file': os.path.basename(p), 'slug': p.split(os.sep)[1], 'name': p.split(os.sep)[1]} for p in kept]
        if (i // B) % 20 == 0: print(f'  {i}/{len(all_files)}')
vectors = np.concatenate(vecs)
np.savez_compressed('index_full.npz', vectors=vectors, meta=json.dumps(meta))
print('saved index_full.npz:', vectors.shape)

In [ ]:
# 6) Accuracy check (leave-one-out)
slugs = [m['slug'] for m in meta]
by = {}
for i, s in enumerate(slugs): by.setdefault(s, []).append(i)
qs = [i for ids in by.values() if len(ids) >= 2 for i in ids]
random.shuffle(qs); qs = qs[:2000]
top1 = top5 = 0
for qi in qs:
    sims = vectors @ vectors[qi]
    sims[qi] = -1
    rk = np.argsort(-sims)[:5]
    hits = [r for r, i in enumerate(rk) if slugs[i] == slugs[qi]]
    if hits:
        top5 += 1
        if hits[0] == 0: top1 += 1
print(f'Top-1: {100*top1/len(qs):.1f}%   Top-5: {100*top5/len(qs):.1f}%   (n={len(qs)})')

In [ ]:
# 7) Download the two result files to your PC
from google.colab import files
files.download('index_full.npz')
files.download('pill_clip_finetuned.pt')